# Option 1 API and live retrieval checks

**Notebook contract: schema 2.1 · 18 table columns · explicit units.**

Start `uv run uvicorn API.server:app --reload --reload-dir API --port 8002` from this repo. Run all cells to verify schema 2.1, the 11 intake sources, evidence bytes and unknown-location behavior. The arbitrary Worcester point is outside the supplied examples. Unsupported municipal adapters stay explicit.

This notebook verifies site intake only. Removed source IDs are rejected; code review, design/economics and compliance fields cannot be restored by `include_all`.

In [1]:
import hashlib
import json
import os
from pathlib import Path
from time import perf_counter

import polars as pl
import requests
from IPython.display import display

BASE_URL = os.getenv("SITE_API_URL", "http://127.0.0.1:8002").rstrip("/")
POINT = {"latitude": 42.2626, "longitude": -71.8023}
OUTPUT = Path("data/processed/api_demo")
OUTPUT.mkdir(parents=True, exist_ok=True)
REQUIRE_ALL_PROVIDERS = False  # Set True only when requiring complete external provider coverage.
session = requests.Session()
checks = []


def get(path, params=None, expected=200):
    started = perf_counter()
    try:
        response = session.get(BASE_URL + path, params=params, timeout=360)
    except requests.ConnectionError as error:
        raise RuntimeError(
            f"Cannot connect to {BASE_URL}. Start the backend from the repository root: "
            "uv run uvicorn API.server:app --reload --port 8002"
        ) from error
    checks.append({"path": path, "url": response.url, "expected_status": expected,
                   "status": response.status_code, "passed": response.status_code == expected,
                   "seconds": round(perf_counter() - started, 3)})
    if response.status_code != expected:
        raise RuntimeError(f"GET {response.url}: expected HTTP {expected}, got {response.status_code}. {response.text[:500]}")
    return response


## Confirm live retrieval

The file cache is revalidated on every request. `checked_at_utc` records the provider check, while `fetched_at_utc` records byte retrieval. HTTP 304 means the provider confirmed unchanged bytes. A changed file at the same URL is downloaded and preserved as a new version. Discovery failures are explicit; there is no fixed-year or fixed-file fallback. Publisher reports can legitimately remain old until the publisher releases an update.

## Check the running API before retrieving data

The API advertises its schema and table columns in `/sources`. This check reports a notebook/API contract mismatch before retrieving site data. If an open tab still expects schema 1.4 or a `match_method` table column, reload this notebook from disk; restarting the kernel does not refresh cell text. No local backend imports are used, so a previously imported Python module cannot set the expected columns.


In [2]:
assert get("/health").json() == {"status": "ok"}
schema = get("/openapi.json").json()
catalog = get("/sources").json()
expected_columns = ['checklist item', 'subcategory_name', 'subcategory_value', 'value_unit_of_measurement', 'source_title', 'fetched_at_utc', 'checked_at_utc', 'source_date', 'source_date_basis', 'searched_latitude', 'searched_longitude', 'source_latitude', 'source_longitude', 'source_distance_m', 'source_location_basis', 'data_source', 'local_file', 'file_url']
required_columns = set(expected_columns)
server_columns = catalog.get("table_columns", [])
missing_columns = required_columns - set(server_columns)
api_version = schema.get("info", {}).get("version", "unknown")
expected_schema = "2.1"
if catalog.get("schema_version") != expected_schema or missing_columns:
    raise RuntimeError(
        f"Notebook/API contract mismatch at {BASE_URL}: notebook expects schema {expected_schema}; "
        f"server advertises schema {catalog.get('schema_version', 'not advertised')} "
        f"(API {api_version}). Missing table columns: {sorted(missing_columns)}. "
        "Reload site_api_demo.ipynb from disk to load the current cell text, then restart the kernel. "
        "If the server still differs from the current notebook contract, check SITE_API_URL "
        "and restart the backend from this repository with: "
        "uv run uvicorn API.server:app --reload --reload-dir API --port 8002."
    )
expected_sources = {"utilities", "parcels", "buildings", "permits", "lber", "tariffs"}
assert expected_sources <= set(catalog["automated"]), f"Unexpected provider catalog at {BASE_URL}: {list(catalog['automated'])}"
assert {"/site", "/site/table", "/evidence/{file_path}"} <= set(schema["paths"])
get("/docs")
get("/redoc")
print(f"Connected to {BASE_URL} — API {api_version}, schema {catalog['schema_version']}, {len(server_columns)} table columns")
display(pl.from_dicts([{"source": name, "title": value["title"], "match_method": value["match_method"]} for name, value in catalog["automated"].items()]))

assert catalog["scope"] == "option_1_site_intake"
assert len(catalog["automated"]) == 11
assert server_columns == expected_columns, "CSV/table columns are out of order."

Connected to http://127.0.0.1:8002 — API 2.1.0, schema 2.1, 18 table columns


source,title,match_method
str,str,str
"""utilities""","""MassGIS — electricity provider…","""municipality_containing_point"""
"""parcels""","""MassGIS — Massachusetts assess…","""point_intersection"""
"""buildings""","""MassGIS — building structures …","""point_intersection"""
"""lber""","""Massachusetts DOER — multi-yea…","""within_250m_candidate"""
"""permits""","""City of Cambridge — Permit Fin…","""within_250m_candidate"""
…,…,…
"""solar_permits""","""City of Cambridge — solar inst…","""within_250m_candidate"""
"""assessor_records""","""Municipal public assessor card…","""parcel_address_candidate"""
"""municipal_archives""","""Municipal public archive searc…","""address_search_candidate"""


## Full public lookup and table

The JSON includes `public_coverage`, explicit missing public/engineering facts and downloadable evidence. `download_evidence=False` skips the exported evidence package; required source files (for example PTS spreadsheets and tariff PDFs) are still cached for extraction.

In [3]:
site = get("/site", POINT).json()
assert site["schema_version"] == "2.1"
assert set(site["sources"]) == set(catalog["default_sources"])
(OUTPUT / "site.json").write_text(json.dumps(site, indent=2))
rows = get("/site/table", {**POINT, "download_evidence": False}).json()
if not isinstance(rows, list) or not rows:
    raise RuntimeError(f"Expected non-empty table rows from {BASE_URL}/site/table; received {str(rows)[:300]}")
missing = [(index, sorted(set(server_columns) - row.keys())) for index, row in enumerate(rows) if set(server_columns) - row.keys()]
if missing:
    raise RuntimeError(f"Table does not match the advertised schema at {BASE_URL}. First incomplete row: {missing[0]}. Restart the API and rerun from the top.")
table = pl.from_dicts(rows, infer_schema_length=None).select(server_columns)
assert table["searched_latitude"].eq(POINT["latitude"]).all()
assert table["searched_longitude"].eq(POINT["longitude"]).all()
table.write_csv(OUTPUT / "checklist.csv")
table.write_parquet(OUTPUT / "checklist.parquet")
with pl.Config(tbl_cols=-1, fmt_str_lengths=80):
    display(table.select("subcategory_name", "subcategory_value", "value_unit_of_measurement", "searched_latitude", "searched_longitude", "source_latitude", "source_longitude", "source_distance_m").head(20))
print("Status:", site["status"], "Provider errors:", site["source_errors"], "Download errors:", site["evidence"]["errors"])

display(pl.from_dicts(site["public_coverage"]).select("source", "status", "records", "truncated", "detail"))
assert site["input_policy"]["uploaded_files_used"] is False
assert site["input_policy"]["manual_document_input"] is False
assert table["value_unit_of_measurement"].is_not_null().all()

subcategory_name,subcategory_value,value_unit_of_measurement,searched_latitude,searched_longitude,source_latitude,source_longitude,source_distance_m
str,str,str,f64,f64,f64,f64,f64
"""address""","""455 MAIN ST""","""not_applicable""",42.2626,-71.8023,42.262513,-71.801582,59.88
"""municipality""","""WORCESTER""","""not_applicable""",42.2626,-71.8023,null,null,null
"""state""","""MA""","""not_applicable""",42.2626,-71.8023,null,null,null
"""latitude""","""42.2626""","""degrees (WGS84)""",42.2626,-71.8023,42.2626,-71.8023,0.0
"""longitude""","""-71.8023""","""degrees (WGS84)""",42.2626,-71.8023,42.2626,-71.8023,0.0
…,…,…,…,…,…,…,…
"""assessor_records""",null,"""not_applicable""",42.2626,-71.8023,null,null,null
"""assessor_records.coverage""","""No public assessor adapter configured for this municipality.""","""not_applicable""",42.2626,-71.8023,null,null,null
"""land_records[1].DISTRICT""","""Worcester""","""not_applicable""",42.2626,-71.8023,null,null,null


Status: partial Provider errors: ['permits', 'electrical_permits', 'solar_permits', 'assessor_records', 'municipal_archives', 'land_records', 'tariffs'] Download errors: []


source,status,records,truncated,detail
str,str,i64,bool,str
"""utilities""","""ok""",1,false,null
"""parcels""","""ok""",1,false,null
"""buildings""","""no_data""",0,false,null
"""lber""","""ok""",45,false,null
"""permits""","""unsupported""",0,false,"""This permit adapter covers Cam…"
…,…,…,…,…
"""solar_permits""","""unsupported""",0,false,"""This permit adapter covers Cam…"
"""assessor_records""","""unsupported""",0,false,"""No public assessor adapter con…"
"""municipal_archives""","""unsupported""",0,false,"""Municipal archive adapter curr…"


## Test every public source independently

Sources needing an address or parcel automatically include the parcel lookup. Utilities always resolves the municipality. `no_data` means an empty published query, not proof that a physical feature or constraint is absent. `partial`, `blocked`, `unsupported` and `error` remain visible below.

In [4]:
provider_results = {}
for name in catalog["automated"]:
    payload = get("/site", {**POINT, "sources": name, "download_evidence": False}).json()
    result = payload["sources"][name]
    provider_results[name] = result
    print(name, result["status"], "records:", len(result["records"]), result.get("error", ""))
    if result["records"]:
        display(pl.from_dicts([{"record": index, **record["_location"]} for index, record in enumerate(result["records"][:10], 1)], infer_schema_length=None))
summary = pl.from_dicts([{"source": name, "status": result["status"], "records": len(result["records"]),
                        "retrieved_at_utc": result["retrieved_at_utc"], "detail": result.get("error") or result.get("detail") or result.get("limitations")} for name, result in provider_results.items()])
summary.write_csv(OUTPUT / "provider_checks.csv")
display(summary)
if REQUIRE_ALL_PROVIDERS:
    assert summary["status"].is_in(["ok", "no_data"]).all(), "Public coverage is incomplete; inspect provider detail above."


utilities ok records: 1 


record,searched_latitude,searched_longitude,source_latitude,source_longitude,source_distance_m,source_location_basis,match_method
i64,f64,f64,null,null,null,str,str
1,42.2626,-71.8023,null,null,null,"""Municipality-wide record; no s…","""municipality_containing_point"""


parcels ok records: 1 


record,searched_latitude,searched_longitude,source_latitude,source_longitude,source_distance_m,source_location_basis,match_method
i64,f64,f64,f64,f64,f64,str,str
1,42.2626,-71.8023,42.262513,-71.801582,59.88,"""Provider polygon centroid in W…","""point_intersection"""


buildings no_data records: 0 


lber ok records: 45 


... (6 outputs left out)

land_records partial records: 1 


record,searched_latitude,searched_longitude,source_latitude,source_longitude,source_distance_m,source_location_basis,match_method
i64,f64,f64,null,null,null,str,str
1,42.2626,-71.8023,null,null,null,"""Source location not supplied""","""district_containing_point"""


tariffs partial records: 0 


source,status,records,retrieved_at_utc,detail
str,str,i64,str,str
"""utilities""","""ok""",1,"""2026-11-07T23:07:35.550864+00:…",null
"""parcels""","""ok""",1,"""2026-11-07T23:07:35.866996+00:…",null
"""buildings""","""no_data""",0,"""2026-11-07T23:07:36.198525+00:…",null
"""lber""","""ok""",45,"""2026-11-07T23:07:37.901330+00:…",null
"""permits""","""unsupported""",0,null,"""This permit adapter covers Cam…"
…,…,…,…,…
"""solar_permits""","""unsupported""",0,null,"""This permit adapter covers Cam…"
"""assessor_records""","""unsupported""",0,null,"""No public assessor adapter con…"
"""municipal_archives""","""unsupported""",0,null,"""Municipal archive adapter curr…"


## Verify local evidence download URLs

`file_url` is relative to the API base URL. Each searched coordinate has its own `data/evidence/<latitude>_<longitude>/` folder, with downloaded files, immutable runs, a latest manifest and a readable index. These URLs serve only public evidence from that folder. Each response is checked against the saved SHA-256 hash. A selected PDF and Jupyter-relative file links are demonstrated in the short notebook.


Every lookup revalidates files; metadata distinguishes a new download from HTTP 304 Not Modified. A manifest contains hashes, extraction limits and the whole parcel-envelope imagery tile coverage. All successful returned file URLs are fetched and their bytes checked below.

In [5]:
for asset in site["evidence"]["files"]:
    if asset["status"] == "ok":
        response = get(asset["file_url"])
        assert hashlib.sha256(response.content).hexdigest() == asset["sha256"]
        print(asset["source_title"], "→", BASE_URL + asset["file_url"])
if site["evidence"]["errors"]:
    display(pl.from_dicts([{ "source": x["source_url"], "error": x["error"] } for x in site["evidence"]["errors"]]))

for name, export in site["evidence"]["exports"].items():
    get(export["file_url"])
    print(name, "→", BASE_URL + export["file_url"])
get(site["evidence"]["index_url"])


MassGIS — electricity providers by municipality → http://127.0.0.1:8002/evidence/42.262600_-71.802300/runs/20261107T230731304399Z/utilities.json
MassGIS — Massachusetts assessor parcels → http://127.0.0.1:8002/evidence/42.262600_-71.802300/runs/20261107T230731304399Z/parcels.json
MassGIS — building structures and roof outlines → http://127.0.0.1:8002/evidence/42.262600_-71.802300/runs/20261107T230731304399Z/buildings.json
Massachusetts DOER — multi-year building energy disclosures → http://127.0.0.1:8002/evidence/42.262600_-71.802300/runs/20261107T230731304399Z/lber.json
City of Cambridge — Permit Finder → http://127.0.0.1:8002/evidence/42.262600_-71.802300/runs/20261107T230731304399Z/permits.json
City of Cambridge — detailed electrical permits → http://127.0.0.1:8002/evidence/42.262600_-71.802300/runs/20261107T230731304399Z/electrical_permits.json
City of Cambridge — solar installation permits → http://127.0.0.1:8002/evidence/42.262600_-71.802300/runs/20261107T230731304399Z/solar_perm

<Response [200]>

## Another facility and expected error responses

The same field names apply at a Fall River site. Automatic permit search currently covers Cambridge; the API explicitly returns a manual portal reference or blocked automation elsewhere. Unavailable fields stay null.


In [6]:
second = get("/site", {"latitude": 41.762125, "longitude": -71.100125, "sources": "parcels,buildings,lber", "download_evidence": False}).json()
assert [[f["name"] for f in c["fields"]] for c in second["checklist"]] == [[f["name"] for f in c["fields"]] for c in site["checklist"]]
for params in [{**POINT, "sources": "public_resources"}, {**POINT, "sources": "masscec"}, {"longitude": -71}, {"latitude": 91, "longitude": -71}, {**POINT, "sources": "unknown"}, {"latitude": 40.7, "longitude": -74}]:
    get("/site", params, expected=422)
get("/evidence/nonexistent.pdf", expected=404)
get("/evidence/%2e%2e/%2e%2e/.env", expected=404)
http_checks = pl.from_dicts(checks)
http_checks.write_csv(OUTPUT / "http_checks.csv")
display(http_checks)
assert http_checks["passed"].all()
print("All HTTP, coordinate and evidence-link checks passed.")


path,url,expected_status,status,passed,seconds
str,str,i64,i64,bool,f64
"""/health""","""http://127.0.0.1:8002/health""",200,200,true,0.021
"""/openapi.json""","""http://127.0.0.1:8002/openapi.…",200,200,true,0.003
"""/sources""","""http://127.0.0.1:8002/sources""",200,200,true,0.001
"""/docs""","""http://127.0.0.1:8002/docs""",200,200,true,0.001
"""/redoc""","""http://127.0.0.1:8002/redoc""",200,200,true,0.001
…,…,…,…,…,…
"""/site""","""http://127.0.0.1:8002/site?lat…",422,422,true,0.002
"""/site""","""http://127.0.0.1:8002/site?lat…",422,422,true,0.002
"""/site""","""http://127.0.0.1:8002/site?lat…",422,422,true,0.96


All HTTP, coordinate and evidence-link checks passed.


Run `uv run python scripts/check_links.py` to verify external provider queries, reference pages and local files from the saved short-notebook response. The CSV report is written to `data/processed/link_checks.csv`. Live source and portal availability can change, so failures remain visible.


In [7]:
first = site["sources"]["land_records"].get("artifacts", [])
second = provider_results["land_records"].get("artifacts", [])
if first and second:
    display(pl.from_dicts([{ "request": label, "checked_at_utc": a["checked_at_utc"],
        "fetched_at_utc": a["fetched_at_utc"], "cache_status": a["cache_status"], "sha256": a["sha256"]
    } for label, a in [("full lookup", first[0]), ("later independent lookup", second[0])]]))
    assert second[0]["checked_at_utc"] > first[0]["checked_at_utc"], "Provider was not checked again."
else:
    print("Public registry data could not be checked; inspect source errors. No stale-file fallback is used.")

request,checked_at_utc,fetched_at_utc,cache_status,sha256
str,str,str,str,str
"""full lookup""","""2026-11-07T23:07:30.921208+00:…","""2026-11-07T22:42:53.271861+00:…","""revalidated_not_modified""","""f0b89f42bb879e24e0443e64d366e5…"
"""later independent lookup""","""2026-11-07T23:07:39.379860+00:…","""2026-11-07T22:42:53.271861+00:…","""revalidated_not_modified""","""f0b89f42bb879e24e0443e64d366e5…"
